# Forecasting: SBA vs a time-series foundation model

Does a foundation model beat classical intermittent-demand methods on warung SKUs?

Run A's prior says **no** — classical wins on short, sparse, lumpy series. This tests it.

```bash
pip install pandas statsforecast chronos-forecasting torch matplotlib
```

Two things are measured, and they can disagree:

1. **Forecast error** (MASE / RMSSE) — the usual answer.
2. **Inventory outcome** — each forecast drives a reorder policy; we count stockouts
   and stock held. This is what the shop actually feels, and it rewards *calibrated
   uncertainty*, not just an accurate point estimate.

A method can lose on (1) and win on (2). That asymmetry is the interesting result.


In [ ]:
import sys, subprocess, warnings, collections, math
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

REPO = Path.cwd()
while not (REPO/'ml'/'generate_sales.py').exists() and REPO != REPO.parent: REPO = REPO.parent
sys.path.insert(0, str(REPO/'ml'))
print('repo:', REPO)

# Fixed categorical order — assigned per method, never cycled.
PALETTE = {'SeasonalNaive':'#2a78d6','AutoETS':'#eb6834','CrostonSBA':'#1baf7a',
           'TSB':'#eda100','Chronos':'#e87ba4'}
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.grid':True,'grid.alpha':.25,'font.size':9})

## 1 · Generate the world

200 SKUs × 180 days. Products, prices and demand archetypes come from the real nota
corpus — archetype is assigned by how often each product actually appears in receipts,
so fast movers are things warungs genuinely restock often.


In [ ]:
from generate_sales import generate, load_catalogue, classify
from simulate_inventory import simulate

DAYS, N_SKU, SEED = 180, 200, 42
catalogue = load_catalogue(REPO/'ml'/'sku_catalogue.csv', N_SKU)
sales  = generate(days=DAYS, seed=SEED, catalogue=catalogue)
purch, ledger = simulate(sales, seed=SEED, catalogue=catalogue)
led = pd.DataFrame(ledger)
print(f'{len(sales):,} demand lines · {len(purch):,} purchases · {len(led):,} ledger rows')
led.head(3)

### The censoring choice

A real system observes `sold`, not `demand` — when stock hits zero the shop records a
sale of zero, not the demand it could not serve. We forecast on **`sold`**, because that
is what SNAPTOCK will actually see. The gap is printed below so the bias is explicit
rather than invisible.


In [ ]:
panel = (led.rename(columns={'date':'ds','sku_id':'unique_id','sold':'y'})
            [['unique_id','ds','y']].copy())
panel['ds'] = pd.to_datetime(panel['ds'])
panel['unique_id'] = panel['unique_id'].astype(str)

lost = led['lost'].sum(); dem = led['demand'].sum()
print(f'demand {dem:,}   sold {led.sold.sum():,}   unmet {lost:,}  ({lost/dem:.1%} censored)')
print(f'panel: {panel.unique_id.nunique()} series x {panel.ds.nunique()} days')

## 2 · What kind of demand is this?

Syntetos–Boylan quadrants. ADI = average interval between demands; CV² = squared
coefficient of variation of demand sizes. Cutoffs 1.32 and 0.49. Routing by quadrant
is the classical strategy we are testing.


In [ ]:
rows=[]
for uid, g in panel.groupby('unique_id'):
    adi, cv2, label = classify(list(g.sort_values('ds').y))
    rows.append({'unique_id':uid,'adi':min(adi,30),'cv2':cv2,'quadrant':label})
quad = pd.DataFrame(rows).set_index('unique_id')
print(quad.quadrant.value_counts().to_string())

fig, ax = plt.subplots(figsize=(6,4.2))
for lab, sub in quad.groupby('quadrant'):
    ax.scatter(sub.adi, sub.cv2, s=22, alpha=.75, label=f'{lab} (n={len(sub)})')
ax.axvline(1.32, color='#888', lw=1, ls='--'); ax.axhline(0.49, color='#888', lw=1, ls='--')
ax.set_xscale('log'); ax.set_xlabel('ADI  (avg interval between demands, log)')
ax.set_ylabel('CV²  (variability of demand size)')
ax.set_title('Demand pattern quadrants — 200 warung SKUs')
ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

## 3 · Backtest protocol

Hold out the final 28 days. Forecast horizon 14 — supplier lead time plus review
period, i.e. how far ahead a restock decision actually has to see.

Metrics are **MASE** and **RMSSE**, scaled by the in-sample naive error. Not MAPE:
it divides by actual demand, and these series are full of zero-demand days.


In [ ]:
H, HOLDOUT = 14, 28
cutoff = panel.ds.max() - pd.Timedelta(days=HOLDOUT)
train = panel[panel.ds <= cutoff].copy()
test  = panel[panel.ds >  cutoff].copy()
print(f'train {train.ds.min().date()} → {train.ds.max().date()}  ({train.ds.nunique()} days)')
print(f'test  {test.ds.min().date()} → {test.ds.max().date()}  ({test.ds.nunique()} days)')

scale = {}   # in-sample naive MAE per series, the MASE denominator
for uid, g in train.groupby('unique_id'):
    y = g.sort_values('ds').y.values
    d = np.abs(np.diff(y))
    scale[uid] = d.mean() if len(d) and d.mean() > 0 else 1.0

## 4 · Classical baselines


In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, AutoETS, CrostonSBA, TSB

sf = StatsForecast(
    models=[SeasonalNaive(season_length=7), AutoETS(season_length=7),
            CrostonSBA(), TSB(alpha_d=0.2, alpha_p=0.2)],
    freq='D', n_jobs=-1)
fc_stats = sf.forecast(df=train, h=H).reset_index()
fc_stats.head(3)

## 5 · Chronos-Bolt, zero-shot

No training, no fitting — the context window is handed straight to a pretrained model.
It returns **quantiles**, which matters in section 7: safety stock needs a distribution,
and the classical methods only give a point estimate.


In [ ]:
import torch
from chronos import BaseChronosPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pipe = BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-base',
                                           device_map=device, torch_dtype=torch.float32)
print('device:', device)

uids = sorted(train.unique_id.unique(), key=int)
ctx  = [torch.tensor(train[train.unique_id==u].sort_values('ds').y.values, dtype=torch.float32)
        for u in uids]

QL = [0.1,0.5,0.9]
q_all, m_all = [], []
for i in range(0, len(ctx), 64):
    q, m = pipe.predict_quantiles(ctx[i:i+64], prediction_length=H, quantile_levels=QL)
    q_all.append(q); m_all.append(m)
quantiles = torch.cat(q_all).numpy()      # (n_series, H, 3)
print('quantiles:', quantiles.shape)

dates = pd.date_range(cutoff + pd.Timedelta(days=1), periods=H, freq='D')
chr_rows = [{'unique_id':u,'ds':d,'Chronos':max(0.,quantiles[i,h,1]),
             'Chronos_q90':max(0.,quantiles[i,h,2])}
            for i,u in enumerate(uids) for h,d in enumerate(dates)]
fc_chr = pd.DataFrame(chr_rows)

## 6 · Forecast accuracy


In [ ]:
fc = fc_stats.merge(fc_chr, on=['unique_id','ds'], how='outer')
ev = fc.merge(test, on=['unique_id','ds'], how='inner')
METHODS = ['SeasonalNaive','AutoETS','CrostonSBA','TSB','Chronos']

def score(df):
    out={}
    for m in METHODS:
        num_a, num_s, n = 0., 0., 0
        for uid, g in df.groupby('unique_id'):
            s = scale[uid]
            num_a += np.abs(g[m]-g.y).sum()/s
            num_s += ((g[m]-g.y)**2).sum()/(s**2)
            n += len(g)
        out[m] = {'MASE':num_a/n, 'RMSSE':math.sqrt(num_s/n)}
    return pd.DataFrame(out).T

overall = score(ev).sort_values('MASE')
print('OVERALL'); print(overall.round(3).to_string()); print()

ev_q = ev.merge(quad[['quadrant']], left_on='unique_id', right_index=True)
per_q = {q: score(g)['MASE'] for q, g in ev_q.groupby('quadrant')}
by_quad = pd.DataFrame(per_q)
print('MASE BY QUADRANT'); print(by_quad.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,3.8))
o = overall.sort_values('MASE')
bars = axes[0].barh(range(len(o)), o.MASE, color=[PALETTE[m] for m in o.index], height=.62)
axes[0].set_yticks(range(len(o))); axes[0].set_yticklabels(o.index)
axes[0].invert_yaxis(); axes[0].set_xlabel('MASE  (lower is better)')
axes[0].set_title('Overall forecast error')
for i,(m,v) in enumerate(o.MASE.items()):
    axes[0].text(v, i, f'  {v:.3f}', va='center', fontsize=8)   # direct labels

bq = by_quad.reindex(columns=[c for c in ['smooth','erratic','intermittent','lumpy'] if c in by_quad])
x = np.arange(len(bq.columns)); w = .16
for k, m in enumerate(METHODS):
    axes[1].bar(x + k*w, bq.loc[m], w, label=m, color=PALETTE[m])
axes[1].set_xticks(x + 2*w); axes[1].set_xticklabels(bq.columns)
axes[1].set_ylabel('MASE'); axes[1].set_title('By demand quadrant')
axes[1].legend(frameon=False, fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

## 7 · The comparison that matters — inventory outcome

Forecast error is a proxy. What the shop feels is: did I run out, and how much cash is
sitting on the shelf?

Each method's forecast drives the same reorder rule over the holdout window:
reorder point = forecast demand over the lead time + safety stock. Classical methods
get safety stock from residual spread (a normal approximation, the standard practice).
Chronos uses its own **q90** directly — no normality assumed.

Better methods sit **up and to the left**: higher fill rate for less stock held.


In [ ]:
LEAD, Z = 3, 1.28   # 3-day lead time, ~90% service level
true = test.set_index(['unique_id','ds']).y

def run_policy(method, use_q90=False):
    fill_num=fill_den=hold=days=0
    for uid, g in fc.groupby('unique_id'):
        g = g.sort_values('ds')
        pred = g[method].clip(lower=0).values
        lead_dem = pred[:LEAD].sum()
        if use_q90:
            ss = max(0., g['Chronos_q90'].values[:LEAD].sum() - lead_dem)
        else:
            resid = train[train.unique_id==uid].y.values
            ss = Z*resid.std()*math.sqrt(LEAD)
        s_pt = lead_dem + ss
        S    = max(s_pt + pred.mean()*7, s_pt+1)
        on_hand, incoming = S, collections.Counter()
        for t, d in enumerate(g.ds):
            on_hand += incoming.pop(t, 0)
            want = float(true.get((uid, d), 0))
            sold = min(want, on_hand); on_hand -= sold
            fill_num += sold; fill_den += want; hold += on_hand; days += 1
            if on_hand <= s_pt and not incoming:
                incoming[t+LEAD] += max(1., S-on_hand)
    return {'fill_rate':fill_num/max(fill_den,1), 'avg_on_hand':hold/max(days,1)}

res = {m: run_policy(m) for m in METHODS}
res['Chronos (q90)'] = run_policy('Chronos', use_q90=True)
biz = pd.DataFrame(res).T.sort_values('fill_rate', ascending=False)
print(biz.assign(fill_rate=lambda d:(d.fill_rate*100).round(2)).round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6.4,4.4))
for name, r in res.items():
    base = name.split(' ')[0]
    ax.scatter(r['avg_on_hand'], r['fill_rate']*100, s=110,
               color=PALETTE[base], edgecolor='white', linewidth=1.5, zorder=3)
    ax.annotate(name, (r['avg_on_hand'], r['fill_rate']*100),
                textcoords='offset points', xytext=(8,4), fontsize=8)
ax.set_xlabel('average units held  (cash tied up →)')
ax.set_ylabel('fill rate %  (↑ fewer stockouts)')
ax.set_title('Service level vs inventory held — up and left is better')
plt.tight_layout(); plt.show()

## 8 · Reading the result

Three questions to answer from the numbers above, in order of what they change:

1. **Does Chronos beat CrostonSBA on MASE?** If not, Run A's prior holds and you ship the
   classical stack — no model weights, no GPU, no download in `docker compose`.
2. **Does the ranking flip in section 7?** If Chronos loses on MASE but wins on fill rate
   at equal stock, the value is in its *calibrated quantiles*, not its point accuracy —
   and the honest conclusion is 'use SBA for the point forecast, but get safety stock
   from a distribution.'
3. **Does the quadrant table justify routing?** If one method wins everywhere, drop the
   ADI/CV² routing — it is complexity you cannot defend.

### Caveats to carry into the proposal

- The data is synthetic and the archetype parameters are hand-chosen. Re-run with a few
  seeds and a parameter sweep; if the ranking flips, it depends on assumptions you invented.
- Forecasts are fit on censored `sold`, which under-states demand. Re-run on `demand` to
  see how much accuracy the censoring costs — that gap is a finding in itself.
- Zero-shot only. Fine-tuning Chronos on the synthetic panel is a separate experiment,
  and on synthetic data it mostly measures whether it can learn your generator.
